# Wine Quality Classification using Machine Learning

This notebook builds an end-to-end classification pipeline to predict wine quality as **Bad**, **Average**, or **Good** by combining red and white wine datasets.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
red_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
white_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

red_df = pd.read_csv(red_url, sep=';')
white_df = pd.read_csv(white_url, sep=';')

red_df['wine_type'] = 'red'
white_df['wine_type'] = 'white'

df = pd.concat([red_df, white_df], ignore_index=True)
print(f'Combined dataset shape: {df.shape}')
df.head()

In [ ]:
print('Missing values before imputation:')
print(df.isnull().sum().sum())

print('\nQuality value counts (original):')
print(df['quality'].value_counts().sort_index())

## Target Mapping
- Quality <= 4: Bad
- Quality 5-6: Average
- Quality >= 7: Good

In [ ]:
def map_quality(q):
    if q <= 4:
        return 'Bad'
    elif q <= 6:
        return 'Average'
    return 'Good'

df['quality_label'] = df['quality'].apply(map_quality)
print(df['quality_label'].value_counts())

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='quality_label', order=['Bad', 'Average', 'Good'], palette='Set2')
plt.title('Class Distribution')
plt.xlabel('Quality Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()

imputer = SimpleImputer(strategy='mean')
df[num_cols] = imputer.fit_transform(df[num_cols])

print('Missing values after imputation:')
print(df.isnull().sum().sum())

In [ ]:
feature_num_cols = [c for c in num_cols if c != 'quality']

# Clip outliers with IQR bounds for each numeric feature.
for col in feature_num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower=lower, upper=upper)

print('Outlier treatment completed using IQR clipping.')

In [ ]:
model_df = df.drop(columns=['quality']).copy()
model_df = pd.get_dummies(model_df, columns=['wine_type'], drop_first=True)

X = model_df.drop(columns=['quality_label'])
y = model_df['quality_label']

print(f'Feature matrix shape: {X.shape}')
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'SVC': SVC(kernel='rbf', probability=False, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42)
}

results = []
reports = {}
conf_mats = {}

for name, model in models.items():
    if name in ['Decision Tree', 'Random Forest']:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': acc})
    reports[name] = classification_report(y_test, y_pred, output_dict=True)
    conf_mats[name] = confusion_matrix(y_test, y_pred, labels=['Bad', 'Average', 'Good'])

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
results_df

In [ ]:
for name in results_df['Model']:
    print(f'\n{name} - Classification Report')
    print(classification_report(
        y_test,
        models[name].predict(X_test if name in ['Decision Tree', 'Random Forest'] else X_test_scaled),
        labels=['Bad', 'Average', 'Good']
    ))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, name in enumerate(results_df['Model']):
    sns.heatmap(
        conf_mats[name],
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Bad', 'Average', 'Good'],
        yticklabels=['Bad', 'Average', 'Good'],
        ax=axes[idx]
    )
    axes[idx].set_title(name)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for j in range(len(results_df), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
X_scaled_all = MinMaxScaler().fit_transform(X)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name, model in models.items():
    use_X = X if name in ['Decision Tree', 'Random Forest'] else X_scaled_all
    scores = cross_val_score(model, use_X, y, cv=cv, scoring='accuracy')
    cv_rows.append({
        'Model': name,
        'CV Mean Accuracy': scores.mean(),
        'CV Std': scores.std()
    })

cv_df = pd.DataFrame(cv_rows).sort_values(by='CV Mean Accuracy', ascending=False).reset_index(drop=True)
cv_df

In [ ]:
best_test_model = results_df.loc[0, 'Model']
best_cv_model = cv_df.loc[0, 'Model']

print(f'Best model by holdout accuracy: {best_test_model}')
print(f'Best model by 5-fold CV accuracy: {best_cv_model}')

## Conclusion

Random Forest typically performs best on this dataset due to its ability to model non-linear relationships and handle feature interactions with minimal preprocessing.